# RealNVP transaction anomaly detection
This notebook trains RealNVP on normal synthetic transactions.
It uses the same saved dataset and seeds as the VAE experiment.
Higher negative indicates the more unusual transaction.

Need to upload the output zip file from the first vae notebook which is 03a_vae.ipynb

In [1]:
from pathlib import Path
import zipfile
import json
import hashlib
import random
import time
import platform

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from google.colab import files

assert torch.cuda.is_available(), "Select a T4 GPU runtime first."

project = Path("/content/transaction_comparison")
project.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]
assert len(zip_names) == 1, "Upload the VAE backup ZIP."

# Restore the shared experiment
with zipfile.ZipFile(zip_names[0]) as archive:
    for name in archive.namelist():
        destination = (project / name).resolve()
        assert destination.is_relative_to(project.resolve())
    archive.extractall(project)

del uploaded

with open(project / "data/dataset_config.json") as file:
    data_config = json.load(file)

arrays = np.load(project / "data/shared_arrays.npz")

X_train = arrays["X_train"].astype(np.float32)
X_validation = arrays["X_validation"].astype(np.float32)
y_validation = arrays["y_validation"].astype(int)

validation = pd.read_csv(project / "data/validation.csv")
features = data_config["features"]
seeds = data_config["model_seeds"]
device = torch.device("cuda")

assert (arrays["y_train"] == 0).all()
assert X_train.shape[1] == len(features)
assert len(validation) == len(X_validation)

print("GPU:", torch.cuda.get_device_name(0))
print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Seeds:", seeds)
print("Shared dataset restored. No new data generated.")

Saving pair3_vae_backup.zip to pair3_vae_backup.zip
GPU: Tesla T4
Training shape: (6000, 6)
Validation shape: (2000, 6)
Seeds: [42, 123, 2026]
Shared dataset restored. No new data generated.


### Model and training
RealNVP has six layers that take turns applying affine coupling, and it starts with a Gaussian base.
distribution. It is trained using normal transactions with negative examples.
log-likelihood.
Each seed uses 50 epochs. The alert threshold is the 95th percentile
of normal validation scores, matching the VAE threshold rule.

In [2]:
import math

flow_settings = {
    "input_dim": len(features),
    "hidden_dim": 32,
    "coupling_layers": 6,
    "scale_limit": 1.0,
    "epochs": 50,
    "batch_size": 128,
    "learning_rate": 0.001,
    "seeds": seeds,
    "score": "negative log-likelihood in standardised feature space",
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class AffineCoupling(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask, scale_limit):
        super().__init__()
        self.register_buffer("mask", mask)
        self.scale_limit = scale_limit

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim * 2),
        )

        # Start near the identity transform
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, x):
        fixed = x * self.mask
        raw_scale, shift = self.network(fixed).chunk(2, dim=1)

        editable = 1 - self.mask
        scale = (
            self.scale_limit * torch.tanh(raw_scale) * editable
        )
        shift = shift * editable

        z = fixed + editable * (x * torch.exp(scale) + shift)
        log_det = scale.sum(dim=1)
        return z, log_det

    def inverse(self, z):
        fixed = z * self.mask
        raw_scale, shift = self.network(fixed).chunk(2, dim=1)

        editable = 1 - self.mask
        scale = (
            self.scale_limit * torch.tanh(raw_scale) * editable
        )
        shift = shift * editable

        return fixed + editable * (
            (z - shift) * torch.exp(-scale)
        )


class TransactionRealNVP(nn.Module):
    def __init__(
        self, input_dim, hidden_dim, coupling_layers, scale_limit
    ):
        super().__init__()
        layers = []

        for layer_index in range(coupling_layers):
            mask = (
                (torch.arange(input_dim) + layer_index) % 2
            ).float()

            layers.append(
                AffineCoupling(
                    input_dim, hidden_dim, mask, scale_limit
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        z = x
        log_det = x.new_zeros(x.shape[0])

        for layer in self.layers:
            z, layer_log_det = layer(z)
            log_det = log_det + layer_log_det

        return z, log_det

    def inverse(self, z):
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x

    def log_prob(self, x):
        z, log_det = self(x)
        base_log_prob = -0.5 * (
            z.square() + math.log(2 * math.pi)
        ).sum(dim=1)

        return base_log_prob + log_det


def make_flow():
    return TransactionRealNVP(
        input_dim=flow_settings["input_dim"],
        hidden_dim=flow_settings["hidden_dim"],
        coupling_layers=flow_settings["coupling_layers"],
        scale_limit=flow_settings["scale_limit"],
    ).to(device)


# Check the forward and inverse transformations
set_seed(42)
check_model = make_flow()
check_input = torch.tensor(X_train[:32], device=device)

with torch.inference_mode():
    check_z, _ = check_model(check_input)
    recovered = check_model.inverse(check_z)

assert torch.allclose(
    check_input, recovered, atol=1e-5, rtol=1e-5
)

del check_model, check_input, check_z, recovered
torch.cuda.empty_cache()

print("RealNVP defined. Inverse check passed.")

RealNVP defined. Inverse check passed.


In [4]:
#training
train_tensor = torch.tensor(X_train, dtype=torch.float32)
validation_tensor = torch.tensor(
    X_validation, dtype=torch.float32, device=device
)

training_records = []

for seed in flow_settings["seeds"]:
    set_seed(seed)
    model = make_flow()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=flow_settings["learning_rate"],
    )

    loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=flow_settings["batch_size"],
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )

    history = []

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    for epoch in range(1, flow_settings["epochs"] + 1):
        model.train()
        total_loss = 0.0

        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            loss = -model.log_prob(batch).mean()
            assert torch.isfinite(loss).item(), "Non-finite loss."

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch)

        epoch_loss = total_loss / len(train_tensor)
        history.append({
            "epoch": epoch,
            "negative_log_likelihood": epoch_loss,
        })

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Seed {seed} | epoch {epoch:02d} | "
                f"NLL {epoch_loss:.4f}"
            )

    torch.cuda.synchronize()
    training_seconds = time.perf_counter() - start
    peak_gib = torch.cuda.max_memory_allocated() / 1024**3

    model.eval()

    with torch.inference_mode():
        validation_scores = (
            -model.log_prob(validation_tensor)
        ).cpu().numpy()

        # Check invertibility after training
        check_x = validation_tensor[:32]
        check_z, _ = model(check_x)
        inverse_error = (
            model.inverse(check_z) - check_x
        ).abs().max().item()

    assert np.isfinite(validation_scores).all()
    assert inverse_error < 1e-3, "Inverse check failed."

    normal_scores = validation_scores[y_validation == 0]
    threshold = float(np.quantile(normal_scores, 0.95))

    torch.save(
        {
            "state_dict": {
                key: value.detach().cpu()
                for key, value in model.state_dict().items()
            },
            "settings": flow_settings,
            "seed": seed,
            "threshold": threshold,
        },
        project / f"models/realnvp_seed{seed}.pt",
    )

    pd.DataFrame(history).to_csv(
        project / f"results/realnvp_history_seed{seed}.csv",
        index=False,
    )

    validation_output = validation.copy()
    validation_output["anomaly_score"] = validation_scores
    validation_output["predicted_unusual"] = (
        validation_scores > threshold
    ).astype(int)

    validation_output.to_csv(
        project / f"outputs/realnvp_validation_seed{seed}.csv",
        index=False,
    )

    training_records.append({
        "seed": seed,
        "training_seconds": training_seconds,
        "training_peak_allocated_gpu_gib": peak_gib,
        "threshold": threshold,
        "parameters": sum(
            parameter.numel() for parameter in model.parameters()
        ),
        "inverse_max_error": inverse_error,
    })

    pd.DataFrame(training_records).to_csv(
        project / "results/realnvp_training_summary.csv",
        index=False,
    )

    del model, optimizer, loss, batch, check_x, check_z
    torch.cuda.empty_cache()

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "precision": "float32",
}

config = {
    "settings": flow_settings,
    "environment": environment,
    "training_timing_scope": (
        "Training loop including batch transfers and loss logging; "
        "excludes model setup, validation and file saving."
    ),
    "threshold_method": (
        "95th percentile of normal validation negative log-likelihood; "
        "alert when score is strictly greater."
    ),
    "limitations": [
        "Foundational RealNVP implementation, not a latest SOTA model.",
        "Continuous density is fitted to mixed discrete and continuous features.",
        "Circular time features have a deterministic relationship.",
        "Low density does not always imply a meaningful anomaly.",
        "Results apply only to this synthetic dataset.",
    ],
}

with open(project / "results/realnvp_config.json", "w") as file:
    json.dump(config, file, indent=2)

display(pd.DataFrame(training_records).round(4))
print("Three RealNVP models saved. Test set not yet evaluated.")

Seed 42 | epoch 01 | NLL 8.1298
Seed 42 | epoch 10 | NLL 1.9438
Seed 42 | epoch 20 | NLL 0.6182
Seed 42 | epoch 30 | NLL -0.0877
Seed 42 | epoch 40 | NLL -0.4700
Seed 42 | epoch 50 | NLL -0.1865
Seed 123 | epoch 01 | NLL 8.1232
Seed 123 | epoch 10 | NLL 2.9703
Seed 123 | epoch 20 | NLL 1.2529
Seed 123 | epoch 30 | NLL 0.5786
Seed 123 | epoch 40 | NLL -0.1369
Seed 123 | epoch 50 | NLL -0.6229
Seed 2026 | epoch 01 | NLL 8.1076
Seed 2026 | epoch 10 | NLL 2.3605
Seed 2026 | epoch 20 | NLL 0.1115
Seed 2026 | epoch 30 | NLL -0.5896
Seed 2026 | epoch 40 | NLL -1.1633
Seed 2026 | epoch 50 | NLL -1.4886


,seed,training_seconds,training_peak_allocated_gpu_gib,threshold,parameters,inverse_max_error
0,42,19.7647,0.0176,6.8212,10056,0.0
1,123,20.7420,0.0176,6.9513,10056,0.0
2,2026,21.2918,0.0176,6.7068,10056,0.0


Three RealNVP models saved. Test set not yet evaluated.


### Test evaluation
The three saved RealNVP models are evaluated using their fixed validation
thresholds. Higher negative log-likelihood indicates a more unusual transaction.
The test data and evaluation metrics match the VAE experiment.

In [5]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

test = pd.read_csv(project / "data/test.csv")
X_test = arrays["X_test"].astype(np.float32)
y_test = arrays["y_test"].astype(int)

assert len(test) == len(X_test) == len(y_test)

test_tensor = torch.tensor(
    X_test, dtype=torch.float32, device=device
)
test_records = []

for seed in flow_settings["seeds"]:
    checkpoint = torch.load(
        project / f"models/realnvp_seed{seed}.pt",
        map_location="cpu",
        weights_only=True,
    )

    model = make_flow()
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    threshold = float(checkpoint["threshold"])

    # Warm up
    with torch.inference_mode():
        warmup = model.log_prob(test_tensor[:32])
    del warmup

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    with torch.inference_mode():
        scores_gpu = -model.log_prob(test_tensor)

    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - start
    peak_gib = torch.cuda.max_memory_allocated() / 1024**3

    scores = scores_gpu.cpu().numpy()
    assert np.isfinite(scores).all()

    predictions = (scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    test_records.append({
        "model": "RealNVP",
        "seed": seed,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(
            y_test, scores
        ),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "false_positive_rate": fp / (fp + tn),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "test_batch_seconds": inference_seconds,
        "test_peak_allocated_gpu_gib": peak_gib,
    })

    predictions_table = test.copy()
    predictions_table["anomaly_score"] = scores
    predictions_table["predicted_unusual"] = predictions

    predictions_table.to_csv(
        project / f"outputs/realnvp_test_seed{seed}.csv",
        index=False,
    )

    print(f"Evaluated seed {seed}")

    del model, scores_gpu
    torch.cuda.empty_cache()

flow_test_results = pd.DataFrame(test_records)
flow_test_results.to_csv(
    project / "results/realnvp_test_metrics.csv",
    index=False,
)

display(flow_test_results.round(4))

metrics = [
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
]

display(
    flow_test_results[metrics]
    .agg(["mean", "std"])
    .round(4)
)

print("Saved RealNVP test predictions and metrics.")

Evaluated seed 42
Evaluated seed 123
Evaluated seed 2026


,model,seed,threshold,roc_auc,average_precision,precision,recall,f1,false_positive_rate,tn,fp,fn,tp,test_batch_seconds,test_peak_allocated_gpu_gib
0,RealNVP,42,6.8212,0.9972,0.9848,0.6655,0.985,0.7944,0.0550,1701,99,3,197,0.0044,0.0179
1,RealNVP,123,6.9513,0.9956,0.9821,0.7004,0.970,0.8134,0.0461,1717,83,6,194,0.0054,0.0179
2,RealNVP,2026,6.7068,0.9891,0.9439,0.6900,0.935,0.7941,0.0467,1716,84,13,187,0.0106,0.0179


,roc_auc,average_precision,precision,recall,f1,false_positive_rate
mean,0.9940,0.9703,0.6853,0.9633,0.8006,0.0493
std,0.0043,0.0229,0.0179,0.0257,0.0111,0.0050


Saved RealNVP test predictions and metrics.


### Findings
RealNVP got an average ROC-AUC score of 0.9940 and an average precision of 0.9703.
At the fixed validation threshold, the mean recall was 96.33%, with a.
4.93% false-positive rate.

Compared to the VAE, RealNVP found more unusual transactions.
generated more false alerts. The VAE achieved higher precision and F1.

These findings apply to the synthetic dataset. Three training seeds
Assess deviations from the initialization state rather than evaluating performance across diverse conditions
datasets.

In [6]:
vae_metrics = pd.read_csv(
    project / "results/vae_test_metrics.csv"
)
flow_metrics = pd.read_csv(
    project / "results/realnvp_test_metrics.csv"
)

combined = pd.concat(
    [vae_metrics, flow_metrics], ignore_index=True
)

assert not combined.duplicated(["model", "seed"]).any()
assert len(combined) == 6

metrics = [
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
]

summary = combined.groupby("model")[metrics].agg(["mean", "std"])

combined.to_csv(
    project / "results/pair3_all_test_metrics.csv",
    index=False,
)
summary.to_csv(
    project / "results/pair3_detection_summary.csv"
)

display(summary.round(4))

# Compare training cost
training_tables = []

for label, filename in [
    ("VAE", "vae_training_summary.csv"),
    ("RealNVP", "realnvp_training_summary.csv"),
]:
    table = pd.read_csv(project / "results" / filename)
    table["model"] = label
    training_tables.append(table)

training = pd.concat(training_tables, ignore_index=True)

cost_summary = training.groupby("model").agg(
    mean_training_seconds=("training_seconds", "mean"),
    mean_training_peak_gpu_gib=(
        "training_peak_allocated_gpu_gib", "mean"
    ),
    parameters=("parameters", "first"),
)

cost_summary.to_csv(
    project / "results/pair3_training_cost_summary.csv"
)

display(cost_summary.round(4))
print("Saved comparison tables.")

roc_auc         average_precision         precision          recall  \
           mean     std              mean     std      mean     std    mean   
model                                                                         
RealNVP  0.9940  0.0043            0.9703  0.0229    0.6853  0.0179  0.9633   
VAE      0.9911  0.0020            0.9468  0.0093    0.7120  0.0287  0.9433   

                     f1         false_positive_rate          
            std    mean     std                mean     std  
model                                                        
RealNVP  0.0257  0.8006  0.0111              0.0493  0.0050  
VAE      0.0104  0.8112  0.0155              0.0426  0.0065

,mean_training_seconds,mean_training_peak_gpu_gib,parameters
model,,,
RealNVP,20.5995,0.0176,10056
VAE,8.8689,0.0170,2860


Saved comparison tables.


In [7]:
import shutil
from google.colab import files

backup = shutil.make_archive(
    "/content/pair3_both_models_backup",
    "zip",
    root_dir=str(project),
)

files.download(backup)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>